# PolyMint — Train polymer + purity classifiers (Colab, free GPU)

End-to-end: TACO real data → YOLOv8n-cls training → float32 TFLite export.
Runtime → Change runtime type → **GPU (T4)** before running.

This mirrors the local `ml/` pipeline; use it when you don't want to install
the heavy stack locally. Download the `.tflite` + labels at the end and drop
them into the Flutter app (see `android/preprocessing_spec.md`).

In [ ]:
!pip -q install ultralytics==8.3.0 tensorflow==2.16.1 onnx onnxruntime onnx2tf onnx_graphsurgeon sng4onnx scikit-learn pyyaml
import torch; print('CUDA:', torch.cuda.is_available())

## 1. Get the ML code + TACO annotations

In [ ]:
# Option A: clone your repo (has the ml/ folder)
!git clone https://github.com/Chandan-GS/poly-mint.git
%cd poly-mint/ml
# TACO annotations (images are downloaded by the script below)
!git clone --depth 1 https://github.com/pedropro/TACO.git data/taco_repo

## 2. Build the real polymer dataset from TACO

In [ ]:
!python src/download_taco.py --ann data/taco_repo/data/annotations.json \
    --map configs/taco_map.yaml --out data/taco_crops --pad 0.15 --min-size 40
!python src/prepare_dataset.py --source data/taco_crops --identity-map \
    --out data/polymer --classes PET HDPE PVC LDPE PP PS Other

## 3. Train (GPU) + export to float32 TFLite

In [ ]:
!python src/train.py --config configs/polymer.yaml --epochs 100 --device 0
!python src/export_tflite.py --weights runs/polymer/weights/best.pt --imgsz 224

## 4. Purity grader
No public purity dataset exists. Upload your own `clean/ mixed/ contaminated/`
images (or use synthetic to test plumbing), then train the second model.

In [ ]:
# Plumbing test with synthetic purity data (replace with real labelled images):
!python src/make_synthetic_dataset.py --out data/purity --classes clean mixed contaminated --per-class 60
!python src/train.py --config configs/purity.yaml --epochs 40 --device 0
!python src/export_tflite.py --weights runs/purity/weights/best.pt --imgsz 224

## 5. Download the artifacts for the app

In [ ]:
from google.colab import files
for f in ['models/polymer_float32.tflite','models/polymer.labels.txt',
          'models/purity_float32.tflite','models/purity.labels.txt']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)